Project 1 Intermediate 📊 Student Analytics System

In [1]:
import random
from collections import defaultdict
from functools import reduce
import math

# -----------------------------
# Generate Student Data
# -----------------------------

random.seed(42)

names = [
    "Amit", "Diya", "Rahul", "Priya", "Neha",
    "Arjun", "Riya", "Karan", "Sneha", "Vikas",
    "Ankit", "Pooja", "Nisha", "Rohan", "Meera"
]

departments = ["Eng", "Mkt", "HR", "Data"]
subjects = ["Maths", "Science", "English", "CS"]


def student_generator(n):
    for _ in range(n):
        name = random.choice(names)
        dept = random.choice(departments)

        yield {
            "name": f"{name}_{dept}",
            "dept": dept,
            "scores": {
                subject: random.randint(50, 99)
                for subject in subjects
            },
            "attendance": random.randint(75, 100),
            "year": random.randint(1, 4)
        }


students = tuple(student_generator(30))


# -----------------------------
# Helper Function
# -----------------------------

def average(student):
    return reduce(
        lambda a, b: a + b,
        student["scores"].values()
    ) / len(subjects)


# -----------------------------
# 1. Top 5 Students
# -----------------------------

print("\nTop 5 Students")

top5 = sorted(
    (
        (s, average(s))
        for s in students
    ),
    key=lambda x: x[1],
    reverse=True
)[:5]

for rank, (student, avg) in enumerate(top5, start=1):
    print(
        f"#{rank:<2}"
        f"{student['name']:<15}"
        f"| Avg: {avg:6.2f}"
        f" | Dept: {student['dept']:<4}"
        f"| Yr:{student['year']}"
    )


# -----------------------------
# 2. Department-wise Average
# -----------------------------

dept_scores = defaultdict(list)

for dept, avg in (
    (s["dept"], average(s))
    for s in students
):
    dept_scores[dept].append(avg)

print("\nDepartment Averages")

for dept, scores in dept_scores.items():
    print(f"{dept:<5}: {sum(scores)/len(scores):6.2f}")


# -----------------------------
# 3. At-risk Students
# -----------------------------

at_risk = tuple(
    s
    for s in students
    if s["attendance"] < 80 and average(s) < 65
)

print(f"\nAt-risk students: {len(at_risk)}")

for student in at_risk:
    print(
        f"{student['name']:<15}"
        f" Avg:{average(student):6.2f}"
        f" Attendance:{student['attendance']}%"
    )


# -----------------------------
# 4. Highest Variance Subject
# -----------------------------

variance = {}

for subject in subjects:
    values = tuple(
        s["scores"][subject]
        for s in students
    )

    mean = sum(values) / len(values)

    var = (
        sum(
            (x - mean) ** 2
            for x in values
        )
        / len(values)
    )

    variance[subject] = math.sqrt(var)

best_subject = max(
    variance.items(),
    key=lambda x: x[1]
)

print(
    f"\nSubject with highest variance: "
    f"{best_subject[0]} (σ={best_subject[1]:.2f})"
)


# -----------------------------
# 5. Year-wise Pass Rate
# -----------------------------

year_stats = defaultdict(lambda: [0, 0])

for student in students:
    year = student["year"]
    year_stats[year][1] += 1

    if average(student) >= 60:
        year_stats[year][0] += 1

print("\nYear-wise Pass Rates")

for year in sorted(year_stats):
    passed, total = year_stats[year]
    rate = passed / total * 100
    print(f"Year {year}: {rate:6.2f}%")


# -----------------------------
# 6. Attendance-Score Correlation
# -----------------------------

attendance = tuple(
    s["attendance"]
    for s in students
)

averages = tuple(
    average(s)
    for s in students
)

mean_att = sum(attendance) / len(attendance)
mean_avg = sum(averages) / len(averages)

cov = sum(
    (a - mean_att) * (b - mean_avg)
    for a, b in zip(attendance, averages)
)

std_att = math.sqrt(
    sum((a - mean_att) ** 2 for a in attendance)
)

std_avg = math.sqrt(
    sum((b - mean_avg) ** 2 for b in averages)
)

correlation = cov / (std_att * std_avg)

print(
    f"\nAttendance-Score correlation: "
    f"{correlation:.2f}"
)


Top 5 Students
#1 Sneha_Mkt      | Avg:  91.00 | Dept: Mkt | Yr:2
#2 Neha_Mkt       | Avg:  90.25 | Dept: Mkt | Yr:2
#3 Priya_HR       | Avg:  87.00 | Dept: HR  | Yr:3
#4 Rahul_HR       | Avg:  85.25 | Dept: HR  | Yr:1
#5 Rohan_Eng      | Avg:  84.75 | Dept: Eng | Yr:4

Department Averages
Eng  :  73.50
Mkt  :  76.32
HR   :  77.31
Data :  74.67

At-risk students: 0

Subject with highest variance: Maths (σ=16.33)

Year-wise Pass Rates
Year 1: 100.00%
Year 2: 100.00%
Year 3: 100.00%
Year 4: 100.00%

Attendance-Score correlation: -0.36


project 2 🎲 Text Adventure Game Engine

In [2]:
import random
from dataclasses import dataclass
from typing import List

random.seed(42)

# -------------------------------------------------
# Custom Exceptions
# -------------------------------------------------

class RoomNotFoundError(Exception):
    pass


class CombatError(Exception):
    pass


class InventoryError(Exception):
    pass


# -------------------------------------------------
# Item Dataclass
# -------------------------------------------------

@dataclass
class Item:
    name: str
    type: str      # weapon / potion / key
    effect_value: int


# -------------------------------------------------
# Enemy Class
# -------------------------------------------------

class Enemy:

    def __init__(self, name, hp, attack):
        self.name = name
        self.hp = hp
        self.attack = attack


# -------------------------------------------------
# Player Class
# -------------------------------------------------

class Player:

    def __init__(self):
        self.hp = 100
        self.inventory: List[Item] = []
        self.position = "Forest Entrance"
        self.score = 0

    def add_item(self, item):
        self.inventory.append(item)

    def has_key(self):
        return any(
            item.type == "key"
            for item in self.inventory
        )

    def save_state(self):
        return {
            "hp": self.hp,
            "position": self.position,
            "score": self.score,
            "inventory": [
                item.__dict__
                for item in self.inventory
            ]
        }

    def load_state(self, state):
        self.hp = state["hp"]
        self.position = state["position"]
        self.score = state["score"]
        self.inventory = [
            Item(**item)
            for item in state["inventory"]
        ]


# -------------------------------------------------
# Game World
# -------------------------------------------------

world = {

    "Forest Entrance": {
        "description": "A peaceful forest.",
        "exits": {
            "north": "Cave",
            "east": "Village"
        },
        "items": [
            Item("Sword", "weapon", 10)
        ],
        "enemy": None
    },

    "Cave": {
        "description": "A dark cave.",
        "exits": {
            "south": "Forest Entrance",
            "east": "Castle Gate"
        },
        "items": [],
        "enemy": Enemy("Goblin", 20, 7)
    },

    "Village": {
        "description": "A quiet village.",
        "exits": {
            "west": "Forest Entrance",
            "north": "Castle Gate"
        },
        "items": [
            Item("Gold Key", "key", 0)
        ],
        "enemy": None
    },

    "Castle Gate": {
        "description": "Huge locked gate.",
        "exits": {
            "north": "Castle",
            "south": "Village",
            "west": "Cave"
        },
        "items": [],
        "enemy": None
    },

    "Castle": {
        "description": "Final castle.",
        "exits": {
            "south": "Castle Gate"
        },
        "items": [
            Item("Health Potion", "potion", 20)
        ],
        "enemy": Enemy("Dragon", 40, 12)
    }

}


# -------------------------------------------------
# Combat System
# -------------------------------------------------

def combat(player, enemy):

    if enemy is None:
        return

    yield f"⚔ Enemy encountered: {enemy.name} (HP:{enemy.hp})"

    while enemy.hp > 0 and player.hp > 0:

        damage = random.randint(5, 12)
        enemy.hp -= damage

        yield f"🗡 You attack for {damage} damage!"

        if enemy.hp <= 0:
            player.score += 50
            yield f"💀 {enemy.name} defeated! +50 score"
            return

        enemy_damage = random.randint(3, enemy.attack)
        player.hp -= enemy_damage

        yield f"💥 {enemy.name} hits you for {enemy_damage}"

    if player.hp <= 0:
        raise CombatError("Player died!")


# -------------------------------------------------
# Generator Event System
# -------------------------------------------------

def play(player, directions):

    for direction in directions:

        room = world.get(player.position)

        if direction not in room["exits"]:
            raise RoomNotFoundError(
                f"No exit {direction}"
            )

        destination = room["exits"][direction]

        yield f"→ Moving {direction} to {destination}..."

        player.position = destination

        room = world[destination]

        # Pick items
        while room["items"]:
            item = room["items"].pop(0)
            player.add_item(item)
            yield f"🎒 Found: {item.name} ({item.type})"

        # Locked Castle
        if destination == "Castle Gate":

            if player.has_key():
                yield "🔒 Door locked! Using Gold Key..."
                yield "✅ Door opened!"
            else:
                raise InventoryError(
                    "Need Gold Key!"
                )

        # Combat
        if room["enemy"]:

            enemy = room["enemy"]

            yield from combat(player, enemy)

            room["enemy"] = None


# -------------------------------------------------
# Main
# -------------------------------------------------

player = Player()

print("=" * 40)
print("      TEXT ADVENTURE GAME")
print("=" * 40)

print(f"\nYou are in: {player.position}")

print("Items here:",
      ", ".join(
          map(
              lambda x: x.name,
              world[player.position]["items"]
          )
      ))

moves = [
    "east",     # Village
    "north",    # Castle Gate
    "west",     # Cave
    "east",     # Castle Gate
    "north"     # Castle
]

print("\nEvents:\n")

try:

    for event in play(player, moves):
        print(event)

except (
    RoomNotFoundError,
    CombatError,
    InventoryError
) as e:

    print(f"Error: {e}")

# Save state
saved = player.save_state()

print("\nSaved State")
print(saved)

# Load state
player.load_state(saved)

print("\nFinal Status")
print(f"HP    : {player.hp}/100")
print(f"Score : {player.score}")

print(
    "Items :",
    list(
        map(
            lambda x: x.name,
            player.inventory
        )
    )
)

print(f"Position: {player.position}")

      TEXT ADVENTURE GAME

You are in: Forest Entrance
Items here: Sword

Events:

→ Moving east to Village...
🎒 Found: Gold Key (key)
→ Moving north to Castle Gate...
🔒 Door locked! Using Gold Key...
✅ Door opened!
→ Moving west to Cave...
⚔ Enemy encountered: Goblin (HP:20)
🗡 You attack for 6 damage!
💥 Goblin hits you for 3
🗡 You attack for 9 damage!
💥 Goblin hits you for 4
🗡 You attack for 8 damage!
💀 Goblin defeated! +50 score
→ Moving east to Castle Gate...
🔒 Door locked! Using Gold Key...
✅ Door opened!
→ Moving north to Castle...
🎒 Found: Health Potion (potion)
⚔ Enemy encountered: Dragon (HP:40)
🗡 You attack for 7 damage!
💥 Dragon hits you for 4
🗡 You attack for 6 damage!
💥 Dragon hits you for 12
🗡 You attack for 11 damage!
💥 Dragon hits you for 3
🗡 You attack for 5 damage!
💥 Dragon hits you for 4
🗡 You attack for 8 damage!
💥 Dragon hits you for 6
🗡 You attack for 5 damage!
💀 Dragon defeated! +50 score

Saved State
{'hp': 64, 'position': 'Castle', 'score': 100, 'inventory': [{'

project 3 📈 Lazy Data Pipeline — Log Analyser

In [3]:
import random
from datetime import datetime, timedelta
from collections import Counter
from itertools import tee
from functools import reduce
import math

random.seed(42)

# =====================================================
# Configuration
# =====================================================

LEVELS = {
    "DEBUG": 40,
    "INFO": 35,
    "WARNING": 15,
    "ERROR": 8,
    "CRITICAL": 2
}

SEVERITY = {
    "DEBUG": 10,
    "INFO": 20,
    "WARNING": 30,
    "ERROR": 40,
    "CRITICAL": 50
}

SERVICES = [
    "UserService",
    "PaymentService",
    "AuthService",
    "DBService",
    "CacheService"
]

MESSAGES = {
    "DEBUG": [
        "Variable initialized",
        "Entering function",
        "Cache refreshed"
    ],
    "INFO": [
        "User logged in",
        "Payment processed",
        "Service started"
    ],
    "WARNING": [
        "High memory usage",
        "Slow response detected",
        "Retry initiated"
    ],
    "ERROR": [
        "Connection timeout",
        "Database unavailable",
        "Request failed"
    ],
    "CRITICAL": [
        "System crash",
        "Data corruption",
        "Authentication failure"
    ]
}

START = datetime(2025, 1, 1)
END = datetime(2025, 3, 31, 23, 59, 59)

# =====================================================
# Lazy Log Generator
# =====================================================

def generate_logs(n=1000):
    total_seconds = int((END - START).total_seconds())

    for _ in range(n):

        ts = START + timedelta(
            seconds=random.randint(0, total_seconds)
        )

        level = random.choices(
            list(LEVELS.keys()),
            weights=LEVELS.values()
        )[0]

        service = random.choice(SERVICES)

        message = random.choice(MESSAGES[level])

        yield (
            f"{ts:%Y-%m-%d %H:%M:%S} "
            f"[{level}] "
            f"{service}: "
            f"{message}"
        )


# =====================================================
# Pipeline Stage 1
# =====================================================

def parse_log(lines):
    for line in lines:

        timestamp = datetime.strptime(
            line[:19],
            "%Y-%m-%d %H:%M:%S"
        )

        level = line.split("[")[1].split("]")[0]

        remaining = line.split("] ")[1]

        service, message = remaining.split(": ", 1)

        yield {
            "timestamp": timestamp,
            "level": level,
            "service": service,
            "message": message
        }


# =====================================================
# Stage 2
# =====================================================

def filter_level(logs, min_level):

    threshold = SEVERITY[min_level]

    for log in logs:
        if SEVERITY[log["level"]] >= threshold:
            yield log


# =====================================================
# Stage 3
# =====================================================

def filter_service(logs, service):

    for log in logs:
        if log["service"] == service:
            yield log


# =====================================================
# Stage 4
# =====================================================

def enrich_log(logs):

    for log in logs:

        log["hour_of_day"] = log["timestamp"].hour
        log["day_of_week"] = log["timestamp"].strftime("%A")

        yield log


# =====================================================
# Stage 5
# =====================================================

def detect_bursts(logs, window=60):

    previous = None

    for log in logs:

        if previous is None:

            log["burst"] = False

        else:

            diff = (
                log["timestamp"] -
                previous["timestamp"]
            ).total_seconds()

            log["burst"] = diff <= window

        previous = log

        yield log


# =====================================================
# Stage 6
# =====================================================

def format_log(logs):

    for log in logs:

        burst = "BURST" if log["burst"] else ""

        yield (
            f"{log['timestamp']} | "
            f"{log['level']:<8} | "
            f"{log['service']:<15} | "
            f"{burst:<5} | "
            f"{log['message']}"
        )


# =====================================================
# Build Lazy Pipeline
# =====================================================

logs = generate_logs()

pipeline = detect_bursts(
    enrich_log(
        parse_log(logs)
    )
)

# Materialize ONCE
records = sorted(
    pipeline,
    key=lambda x: x["timestamp"]
)

formatted = format_log(iter(records))

# =====================================================
# Analytics
# =====================================================

print("=" * 50)
print("LOG ANALYSER")
print("=" * 50)

print(f"Generated {len(records)} log entries (lazily)\n")

# -----------------------------------------------------
# Level Distribution
# -----------------------------------------------------

print("--- Level Distribution ---")

level_counter = Counter(
    log["level"]
    for log in records
)

for level in LEVELS:

    count = level_counter[level]

    print(
        f"{level:<8}: "
        f"{count:4d} "
        f"({count/len(records)*100:5.1f}%)"
    )

# -----------------------------------------------------
# Errors By Hour
# -----------------------------------------------------

print("\n--- Errors by Hour ---")

hour_counter = Counter(
    log["hour_of_day"]
    for log in records
    if log["level"] in ("ERROR", "CRITICAL")
)

peak_hour, peak_count = hour_counter.most_common(1)[0]

print(
    f"Peak error hour: "
    f"{peak_hour:02d}:00 "
    f"({peak_count} errors)"
)

# -----------------------------------------------------
# Problematic Service
# -----------------------------------------------------

print("\n--- Most Problematic Service ---")

service_counter = Counter(
    log["service"]
    for log in records
    if log["level"] in ("ERROR", "CRITICAL")
)

service, count = service_counter.most_common(1)[0]

print(f"{service}: {count} errors/criticals")

# -----------------------------------------------------
# Burst Report
# -----------------------------------------------------

print("\n--- Burst Detection ---")

burst_count = sum(
    log["burst"]
    for log in records
)

print(f"Burst events detected: {burst_count}")

# -----------------------------------------------------
# Average Time Between Errors
# -----------------------------------------------------

print("\n--- Avg Time Between Errors ---")

errors = [
    log["timestamp"]
    for log in records
    if log["level"] in ("ERROR", "CRITICAL")
]

if len(errors) > 1:

    gaps = [
        (b - a).total_seconds()
        for a, b in zip(errors, errors[1:])
    ]

    avg_gap = reduce(
        lambda x, y: x + y,
        gaps
    ) / len(gaps)

    hours = math.floor(avg_gap / 3600)
    minutes = math.floor((avg_gap % 3600) / 60)

    print(
        f"Average gap: "
        f"{hours}h {minutes}m"
    )

else:

    print("Not enough error logs.")

# -----------------------------------------------------
# Sample Formatted Logs
# -----------------------------------------------------

print("\n--- Sample Logs ---")

for _, log in zip(range(10), formatted):
    print(log)

LOG ANALYSER
Generated 1000 log entries (lazily)

--- Level Distribution ---
DEBUG   :  403 ( 40.3%)
INFO    :  357 ( 35.7%)
ERROR   :   80 (  8.0%)
CRITICAL:   24 (  2.4%)

--- Errors by Hour ---
Peak error hour: 14:00 (8 errors)

--- Most Problematic Service ---
PaymentService: 27 errors/criticals

--- Burst Detection ---
Burst events detected: 505

--- Avg Time Between Errors ---
Average gap: 20h 8m

--- Sample Logs ---
2025-01-01 02:59:07 | DEBUG    | DBService       | BURST | Cache refreshed
2025-01-01 03:24:24 | WARNING  | DBService       | BURST | Retry initiated
2025-01-01 03:38:42 | INFO     | PaymentService  | BURST | Service started
2025-01-01 06:14:05 | INFO     | CacheService    | BURST | Payment processed
2025-01-01 08:15:43 | DEBUG    | PaymentService  | BURST | Cache refreshed
2025-01-01 12:06:20 | ERROR    | DBService       | BURST | Database unavailable
2025-01-01 14:51:41 | ERROR    | UserService     | BURST | Connection timeout
2025-01-01 18:47:13 | INFO     | UserS